# НЕЙРОСЕТЬ: ``` Ranker V6 ```

# Подготовка

### Датасет

In [1]:
import json
from pathlib import Path
import numpy as np
import polars as pl
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

FLOAT_FEATURES = ["f1", "f2", "f3", "f4", "f5"]
CANDIDATE_COLUMNS = ["item_id", "category_id", *FLOAT_FEATURES, "item_id__embedding_idx"]
HISTORY_COLUMNS = [
    "history__item_id",
    "history__category_id",
    "history__event_type",
    "history__event_source",
    "history__item_id__embedding_idx",
]
MAX_HIST_LEN = 48  


def read_feeds_v6(split: Path, max_feeds: int = 0) -> pl.DataFrame:
    source = pl.scan_parquet(sorted((split / "impressions").glob("*.parquet")))
    schema = source.collect_schema()
    extra_cols = [c for c in ("target", "pos") if c in schema]

    source = source.select("feed_id", "timestamp", "user_embedding", *CANDIDATE_COLUMNS, *HISTORY_COLUMNS, *extra_cols)
    if max_feeds:
        source = source.head(max_feeds)
    return source.collect(engine="streaming")


class CandidateDataV6(Dataset):
    def __init__(self, split: Path, max_feeds: int = 0, categories_mapping: dict[int, int] | None = None):
        feeds = read_feeds_v6(split, max_feeds)
        candidate_counts = feeds["item_id"].list.len().to_numpy()
        hist_counts = feeds["history__item_id"].list.len().fill_null(0).to_numpy()
        self.hist_offsets = np.insert(hist_counts.cumsum(), 0, 0)

        # 1. Компактная упаковка истории без лишних строк при пустых списках
        non_empty_hist = feeds.filter(pl.col("history__item_id").list.len() > 0).select(HISTORY_COLUMNS)
        hist_flat = non_empty_hist.explode(HISTORY_COLUMNS)

        self.hist_items = hist_flat["history__item_id"].fill_null(0).to_numpy().astype(np.int32)
        self.hist_raw_cats = hist_flat["history__category_id"].fill_null(0).to_numpy().astype(np.int16)
        self.hist_event_type = hist_flat["history__event_type"].fill_null(0).to_numpy().astype(np.int64)
        self.hist_event_source = hist_flat["history__event_source"].fill_null(0).to_numpy().astype(np.int64)
        self.hist_emb_idx = hist_flat["history__item_id__embedding_idx"].fill_null(-1).to_numpy().astype(np.int32)

        assert len(self.hist_items) == self.hist_offsets[-1], "Ошибка смещений истории!"
        del non_empty_hist, hist_flat

        # 2. Циклические признаки времени из timestamp (час суток и день недели)
        ts = feeds["timestamp"].to_numpy().astype(np.float64)
        hour = (ts % 86400) / 86400.0 * (2 * np.pi)
        dow = ((ts // 86400) % 7) / 7.0 * (2 * np.pi)
        self.time_feats = np.stack([np.sin(hour), np.cos(hour), np.sin(dow), np.cos(dow)], axis=1).astype(np.float32)

        # 3. Разворачиваем кандидатов
        explode_cols = CANDIDATE_COLUMNS + (["target"] if "target" in feeds else []) + (["pos"] if "pos" in feeds else [])
        flat = feeds.drop(["user_embedding", "timestamp", *HISTORY_COLUMNS]).explode(explode_cols)

        self.categories_mapping = (
            {val: idx + 1 for idx, val in enumerate(sorted(flat["category_id"].unique().to_list()))}
            if categories_mapping is None
            else categories_mapping
        )
        category = flat["category_id"].replace_strict(self.categories_mapping, default=0, return_dtype=pl.Int64)
        self.category = torch.tensor(category.to_numpy())

        # Маппим категории истории тем же общим словарём
        hist_mapped_cats = pl.Series(self.hist_raw_cats).replace_strict(
            self.categories_mapping, default=0, return_dtype=pl.Int64
        )
        self.hist_cats = hist_mapped_cats.to_numpy()

        self.raw_cat_id = flat["category_id"].to_numpy().astype(np.int16)
        self.raw_item_id = flat["item_id"].to_numpy().astype(np.int32)
        self.floats = torch.tensor(flat.select(FLOAT_FEATURES).to_numpy(), dtype=torch.float32)
        self.users = F.normalize(torch.tensor(feeds["user_embedding"].to_list(), dtype=torch.float32), dim=1)
        self.targets = torch.tensor(flat["target"].to_numpy(), dtype=torch.float32) if "target" in flat else None
        self.positions = torch.tensor(flat["pos"].to_numpy(), dtype=torch.float32) if "pos" in flat else None

        self.keys = flat.select("feed_id", "item_id")
        self.feed_ids = feeds["feed_id"].to_numpy()
        self.offsets = np.insert(candidate_counts.cumsum(), 0, 0)
        self.item_index = flat["item_id__embedding_idx"].to_numpy()
        del feeds, flat

        meta = json.loads((split / "item_embeddings" / "data.meta").read_text())
        self.items = np.memmap(
            split / "item_embeddings" / "data.memmap",
            mode="r",
            dtype=meta["dtype"],
            shape=tuple(meta["shape"]),
            order=meta["order"],
        )

    def __len__(self) -> int:
        return len(self.users)

    def __getitem__(self, index: int) -> tuple:
        rows = slice(self.offsets[index], self.offsets[index + 1])
        num_cands = self.offsets[index + 1] - self.offsets[index]

        indices = self.item_index[rows]
        missing = indices == -1
        items_raw = self.items[np.maximum(indices, 0)].astype(np.float32)
        items_raw[missing] = 0.0
        items_norm = F.normalize(torch.from_numpy(items_raw), dim=1)

        h_start, h_end = self.hist_offsets[index], self.hist_offsets[index + 1]
        h_len = h_end - h_start

        in_hist_item = np.zeros((num_cands, 1), dtype=np.float32)
        hist_cat_share = np.zeros((num_cands, 1), dtype=np.float32)
        hist_last_sim = torch.zeros((num_cands, 1), dtype=torch.float32)
        hist_max_sim = torch.zeros((num_cands, 1), dtype=torch.float32)

        seq_len = min(h_len, MAX_HIST_LEN)
        if seq_len > 0:
            h_slice_full = slice(h_start, h_end)
            h_slice_seq = slice(h_start, h_start + seq_len)

            h_items_full = self.hist_items[h_slice_full]
            c_items = self.raw_item_id[rows]
            valid_h_items = h_items_full[h_items_full > 0]
            if len(valid_h_items) > 0:
                in_hist_item[:, 0] = np.isin(c_items, valid_h_items).astype(np.float32)

            h_raw_cats_full = self.hist_raw_cats[h_slice_full]
            c_cats = self.raw_cat_id[rows]
            hist_cat_share[:, 0] = (c_cats[:, None] == h_raw_cats_full[None, :]).mean(axis=1)

            h_emb_idx = self.hist_emb_idx[h_slice_seq]
            h_missing = h_emb_idx == -1
            h_items_raw = self.items[np.maximum(h_emb_idx, 0)].astype(np.float32)
            h_items_raw[h_missing] = 0.0
            h_embs_norm = F.normalize(torch.from_numpy(h_items_raw), dim=1)

            valid_mask = ~h_missing
            if valid_mask.any():
                sim_matrix = torch.matmul(items_norm, h_embs_norm[valid_mask].T)
                hist_last_sim[:, 0] = sim_matrix[:, 0]
                hist_max_sim[:, 0] = sim_matrix.max(dim=1).values

            h_cat_seq = torch.from_numpy(self.hist_cats[h_slice_seq].copy())
            h_etype_seq = torch.from_numpy(self.hist_event_type[h_slice_seq]).clamp(min=0, max=30) + 1
            h_esrc_seq = torch.from_numpy(self.hist_event_source[h_slice_seq]).clamp(min=0, max=30) + 1
            h_miss_seq = torch.from_numpy(h_missing)[:, None].float()
        else:
            h_embs_norm = torch.zeros((1, 128), dtype=torch.float32)
            h_cat_seq = torch.zeros((1,), dtype=torch.long)
            h_etype_seq = torch.zeros((1,), dtype=torch.long)
            h_esrc_seq = torch.zeros((1,), dtype=torch.long)
            h_miss_seq = torch.ones((1, 1), dtype=torch.float32)

        t_feats = torch.from_numpy(self.time_feats[index]).unsqueeze(0).expand(num_cands, 4)

        features = torch.cat(
            (
                self.floats[rows],
                items_norm,
                torch.from_numpy(missing)[:, None].float(),
                torch.from_numpy(in_hist_item),
                torch.from_numpy(hist_cat_share),
                hist_last_sim,
                hist_max_sim,
                t_feats,
            ),
            dim=1,
        )

        hist_features = torch.cat((h_embs_norm, h_miss_seq), dim=1)
        has_history = seq_len > 0

        target = self.targets[rows] if self.targets is not None else None
        pos = self.positions[rows] if self.positions is not None else None
        return (
            features,
            self.category[rows],
            self.users[index],
            hist_features,
            h_cat_seq,
            h_etype_seq,
            h_esrc_seq,
            has_history,
            target,
            pos,
            self.feed_ids[index],
        )


def collate_feeds_v6(feeds: list[tuple]) -> tuple:
    (
        features,
        categories,
        users,
        hist_features,
        h_cats,
        h_etypes,
        h_esrcs,
        has_histories,
        targets,
        positions,
        feed_ids,
    ) = zip(*feeds, strict=True)

    cand_counts = torch.tensor([len(x) for x in features])
    cand_padding = torch.arange(cand_counts.max())[None, :] >= cand_counts[:, None]

    hist_counts = torch.tensor([len(x) if has_h else 0 for x, has_h in zip(hist_features, has_histories)])
    max_h = max(1, int(hist_counts.max().item()))
    hist_padding = torch.arange(max_h)[None, :] >= hist_counts[:, None]

    return (
        pad_sequence(features, batch_first=True),
        pad_sequence(categories, batch_first=True),
        torch.stack(users),
        cand_padding,
        pad_sequence(hist_features, batch_first=True),
        pad_sequence(h_cats, batch_first=True),
        pad_sequence(h_etypes, batch_first=True),
        pad_sequence(h_esrcs, batch_first=True),
        hist_padding,
        pad_sequence(targets, batch_first=True) if targets[0] is not None else None,
        pad_sequence(positions, batch_first=True, padding_value=-1.0) if positions[0] is not None else None,
        torch.tensor(feed_ids),
    )


### Функция потерь   ``` PairwiseFeedLoss  ```

In [2]:
class PairwiseFeedLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, logits, targets, padding):
        """
        logits:  [Batch, Max_Len] — выходы модели (Ranker) до сигмоиды
        targets: [Batch, Max_Len] — реальные маркеры (0 или 1)
        padding: [Batch, Max_Len] — True там, где фейковые товары (паддинг)
        """
        total_loss = 0.0
        valid_feeds_count = 0
        
        # Проходим по каждой ленте в батче отдельно
        for i in range(logits.size(0)):
            # Маска реальных элементов в этой ленте (не паддинг)
            real_mask = ~padding[i]
            if not real_mask.any():
                continue
                
            feed_logits = logits[i][real_mask]
            feed_targets = targets[i][real_mask]
            
            # Находим индексы позитивных и негативных примеров в этой ленте
            pos_indices = (feed_targets == 1).nonzero(as_tuple=True)[0]
            neg_indices = (feed_targets == 0).nonzero(as_tuple=True)[0]
            
            # Если в ленте нет или позитивных, или негативных — лосс по ней не посчитать
            if len(pos_indices) == 0 or len(neg_indices) == 0:
                continue
            
            # Формируем все возможные пары (pos, neg) через векторизацию
            pos_scores = feed_logits[pos_indices].unsqueeze(1)  # [Num_Pos, 1]
            neg_scores = feed_logits[neg_indices].unsqueeze(0)  # [1, Num_Neg]
            
            # Мы хотим, чтобы pos_scores > neg_scores + margin
            # Формула штрафа: max(0, margin - (pos_scores - neg_scores))
            pairwise_loss = F.softplus(-(pos_scores - neg_scores) + self.margin)
            
            total_loss += pairwise_loss.mean()
            valid_feeds_count += 1
            
        if valid_feeds_count == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
            
        return total_loss / valid_feeds_count


### Функция потерь ``` LambdaFeedLossV5 ```

In [3]:
class LambdaFeedLossV5(nn.Module):
    def __init__(self, k=30, sigma=1.0, pos_alpha=0.25):
        super().__init__()
        self.k = k
        self.sigma = sigma
        self.pos_alpha = pos_alpha

    def forward(self, logits, targets, padding, pos=None):
        total_loss = 0.0
        valid_feeds = 0

        for i in range(logits.size(0)):
            mask = ~padding[i]
            if not mask.any():
                continue

            s = logits[i][mask]
            y = targets[i][mask]

            pos_idx = (y == 1).nonzero(as_tuple=True)[0]
            neg_idx = (y == 0).nonzero(as_tuple=True)[0]

            if len(pos_idx) == 0 or len(neg_idx) == 0:
                continue

            with torch.no_grad():
                order = torch.argsort(s, descending=True)
                ranks = torch.empty_like(order)
                ranks[order] = torch.arange(len(s), device=s.device)

                ideal_y, _ = torch.sort(y, descending=True)
                ideal_y = ideal_y[: self.k]
                discounts = 1.0 / torch.log2(torch.arange(len(s), device=s.device, dtype=torch.float32) + 2.0)
                idcg = (ideal_y * discounts[: len(ideal_y)]).sum()

                if idcg == 0:
                    continue

                pos_ranks = ranks[pos_idx]
                neg_ranks = ranks[neg_idx]

                pos_disc = torch.where(
                    pos_ranks < self.k, discounts[pos_ranks.clamp(max=self.k - 1)], torch.tensor(0.0, device=s.device)
                )
                neg_disc = torch.where(
                    neg_ranks < self.k, discounts[neg_ranks.clamp(max=self.k - 1)], torch.tensor(0.0, device=s.device)
                )

                delta_ndcg = torch.abs(pos_disc.unsqueeze(1) - neg_disc.unsqueeze(0)) / idcg
                delta_ndcg = torch.clamp(delta_ndcg, max=1.0)  # <--- КЛИППИРОВАНИЕ

                if pos is not None:
                    p = pos[i][mask][pos_idx].clamp(min=0.0)
                    pos_weights = (1.0 + self.pos_alpha * torch.log1p(p)).unsqueeze(1)
                else:
                    pos_weights = 1.0

            pos_scores = s[pos_idx].unsqueeze(1)
            neg_scores = s[neg_idx].unsqueeze(0)

            pairwise_loss = F.softplus(-self.sigma * (pos_scores - neg_scores))
            weighted_loss = (pairwise_loss * delta_ndcg * pos_weights).sum()

            num_pairs = len(pos_idx) * len(neg_idx)  # <--- ВСЕ ПАРЫ
            total_loss += weighted_loss / num_pairs
            valid_feeds += 1

        if valid_feeds == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        return total_loss / valid_feeds


### Ансамбль функции потерь  ``` HybridRankingLossV5 ```

In [4]:

import torch
from torch import nn


class HybridRankingLossV5(nn.Module):
    def __init__(
        self,
        weights: dict[str, float] | None = None,
        margin: float = 0.5,
        k: int = 30,
        sigma: float = 1.0,
    ):
        super().__init__()
        self.weights = weights or {"bce": 0.15, "pairwise": 0.25, "lambda": 0.60}
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.pair_loss = PairwiseFeedLoss(margin=margin)
        self.lambda_loss = LambdaFeedLossV5(k=k, sigma=sigma, pos_alpha=0.25)

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        padding_mask: torch.Tensor,
        pos: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, dict[str, float]]:
        valid_mask = ~padding_mask
        raw_metrics: dict[str, float] = {}
        total_grad_loss = torch.zeros((), device=logits.device, dtype=torch.float32)

        if self.weights.get("bce", 0.0) > 0:
            l_bce = self.bce_loss(logits[valid_mask], targets[valid_mask])
            raw_metrics["bce"] = float(l_bce.detach().item())
            total_grad_loss = total_grad_loss + self.weights["bce"] * torch.log1p(l_bce)

        if self.weights.get("pairwise", 0.0) > 0:
            l_pair = self.pair_loss(logits, targets, padding_mask)
            raw_metrics["pair"] = float(l_pair.detach().item())
            total_grad_loss = total_grad_loss + self.weights["pairwise"] * torch.log1p(l_pair)

        if self.weights.get("lambda", 0.0) > 0:
            l_lmb = self.lambda_loss(logits, targets, padding_mask, pos=pos)
            raw_metrics["lmb"] = float(l_lmb.detach().item())
            total_grad_loss = total_grad_loss + self.weights["lambda"] * torch.log1p(l_lmb)

        return total_grad_loss, raw_metrics



###   метрика ``` NDCG ```

In [5]:
# 
def calculate_ndcg(y_true, y_score, feed_ids, k=30):
    unique_feeds = np.unique(feed_ids)
    ndcg_scores = []
    for feed in unique_feeds:
        mask = (feed_ids == feed)
        y_t, y_s = y_true[mask], y_score[mask]
        order = np.lexsort((y_t, -y_s))
        y_t_sorted = y_t[order][:k]
        dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(y_t_sorted))
        ideal_y = np.sort(y_t)[::-1][:k]
        idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_y))
        ndcg_scores.append(dcg / idcg if idcg > 0 else 1.0)
    return np.mean(ndcg_scores)


### Модель: ``` RankerV6_2 ```

In [6]:
class RankerV6(nn.Module):
    def __init__(self, category_count, width=128, layers=3, dropout=0.15):
        super().__init__()
        self.category = nn.Embedding(category_count + 1, 16, padding_idx=0)
        
        # Эмбеддинги для событий истории
        self.event_type_emb = nn.Embedding(32, 16, padding_idx=0)
        self.event_source_emb = nn.Embedding(32, 8, padding_idx=0)
        self.hist_pos_emb = nn.Embedding(MAX_HIST_LEN, 16)  # Позиция свежести (0 = самое новое)

        # Проекция токена истории: 129 (emb + missing) + 16 (cat) + 16 (type) + 8 (source) + 16 (pos) = 185
        self.hist_proj = nn.Sequential(
            nn.Linear(185, width),
            nn.LayerNorm(width),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Cross-Attention: кандидаты (Query) опрашивают историю пользователя (Key, Value)
        nhead = 8 if width % 8 == 0 else 4
        self.hist_cross_attn = nn.MultiheadAttention(
            embed_dim=width, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.hist_norm = nn.LayerNorm(width)

        # Проекция кандидата: 142 (из CandidateDataV6) + 16 + 1 + 10 + 32 + 5 + 1 + 5 + 2 = 214
        self.interaction_proj = nn.Sequential(
            nn.Linear(128, 32),
            nn.LayerNorm(32),
            nn.GELU(),
        )
        in_features_dim = 142 + 16 + 1 + 10 + 32 + 5 + 1 + 5 + 2  # 214

        self.candidate = nn.Sequential(
            nn.Linear(in_features_dim, width * 2),
            nn.LayerNorm(width * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(width * 2, width),
            nn.LayerNorm(width),
        )

        self.user = nn.Sequential(
            nn.Linear(128, width * 2),
            nn.LayerNorm(width * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(width * 2, width),
            nn.LayerNorm(width),
        )

        # Основной Self-Attention энкодер по кандидатам и пользователю
        layer = nn.TransformerEncoderLayer(
            d_model=width,
            nhead=nhead,
            dim_feedforward=4 * width,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)

        # Выходная голова: [item_context, user_context, feed_mean_context, hist_attn_context] -> width * 4
        self.score = nn.Sequential(
            nn.Linear(width * 4, width),
            nn.LayerNorm(width),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(width, 1),
        )

    def forward(
        self,
        features,
        category,
        users,
        cand_padding,
        hist_features,
        h_cats,
        h_etypes,
        h_esrcs,
        hist_padding,
    ):
        B, L, _ = features.shape
        H = hist_features.size(1)
        mask = ~cand_padding
        mask_f = mask.unsqueeze(-1).float()
        counts = mask_f.sum(dim=1, keepdim=True).clamp(min=1.0)

        f_num = features[:, :, :5]

        # 1. Полентные статистики (Z-score, Min-Max, Ranks, Category-aggregates)
        f_mean = (f_num * mask_f).sum(dim=1, keepdim=True) / counts
        f_var = (((f_num - f_mean) * mask_f) ** 2).sum(dim=1, keepdim=True) / counts
        f_std = f_var.sqrt().clamp(min=1e-6)
        f_zscore = ((f_num - f_mean) / f_std) * mask_f

        f_for_min = f_num.masked_fill(cand_padding.unsqueeze(-1), float("inf"))
        f_for_max = f_num.masked_fill(cand_padding.unsqueeze(-1), float("-inf"))
        f_min = torch.where(torch.isinf(f_for_min.min(dim=1, keepdim=True).values), torch.zeros_like(f_mean), f_for_min.min(dim=1, keepdim=True).values)
        f_max = torch.where(torch.isinf(f_for_max.max(dim=1, keepdim=True).values), torch.zeros_like(f_mean), f_for_max.max(dim=1, keepdim=True).values)
        f_minmax = ((f_num - f_min) / (f_max - f_min).clamp(min=1e-6)) * mask_f

        f_masked_inf = f_num.masked_fill(cand_padding.unsqueeze(-1), float("inf"))
        order = torch.argsort(f_masked_inf, dim=1)
        ranks = torch.zeros_like(f_num)
        src_ranks = torch.arange(L, device=features.device, dtype=torch.float32).view(1, L, 1).expand_as(f_num)
        ranks.scatter_(1, order, src_ranks)
        f_ranks = (ranks / (counts - 1.0).clamp(min=1.0)) * mask_f

        same_cat = (category.unsqueeze(2) == category.unsqueeze(1)) & mask.unsqueeze(2) & mask.unsqueeze(1)
        same_cat_f = same_cat.float()
        cat_counts = same_cat_f.sum(dim=2, keepdim=True).clamp(min=1.0)
        cat_freq = (cat_counts / counts) * mask_f
        cat_f_mean = torch.bmm(same_cat_f, f_num) / cat_counts
        cat_f_diff = ((f_num - cat_f_mean) / f_std) * mask_f

        item_emb = features[:, :, 5:133]
        user_expanded = users.unsqueeze(1)
        user_item_sim = (item_emb * user_expanded).sum(dim=-1, keepdim=True)
        hadamard_prod = self.interaction_proj(item_emb * user_expanded)

        feed_log_len = (torch.log1p(counts) / 5.0).expand(B, L, 1) * mask_f
        feed_mean_sim = ((user_item_sim * mask_f).sum(dim=1, keepdim=True) / counts).expand(B, L, 1) * mask_f

        cand_in = torch.cat(
            (
                features,
                self.category(category),
                user_item_sim,
                f_zscore,
                f_minmax,
                hadamard_prod,
                f_ranks,
                cat_freq,
                cat_f_diff,
                feed_log_len,
                feed_mean_sim,
            ),
            dim=-1,
        )
        candidates = self.candidate(cand_in)  # [B, L, width]

        # 2. Кодируем последовательность истории пользователя
        pos_ids = torch.arange(H, device=features.device).unsqueeze(0).expand(B, H)
        hist_in = torch.cat(
            (
                hist_features,
                self.category(h_cats),
                self.event_type_emb(h_etypes),
                self.event_source_emb(h_esrcs),
                self.hist_pos_emb(pos_ids),
            ),
            dim=-1,
        )
        hist_tokens = self.hist_proj(hist_in)  # [B, H, width]

        # Cross-Attention с защитой от полностью пустой истории в батче
        all_hist_masked = hist_padding.all(dim=1, keepdim=True)  # [B, 1]
        safe_hist_padding = hist_padding.clone()
        safe_hist_padding[all_hist_masked.squeeze(1), 0] = False

        hist_context, _ = self.hist_cross_attn(
            query=candidates,
            key=hist_tokens,
            value=hist_tokens,
            key_padding_mask=safe_hist_padding,
        )
        hist_context = hist_context.masked_fill(all_hist_masked.unsqueeze(-1), 0.0)
        hist_context = self.hist_norm(hist_context)

        enriched_candidates = candidates + hist_context

        # 3. Основной Transformer Encoder по ленте кандидатов
        user_token = self.user(users)[:, None, :]
        tokens = torch.cat((user_token, enriched_candidates), dim=1)
        attn_mask = F.pad(cand_padding, (1, 0), value=False)

        context = self.encoder(tokens, src_key_padding_mask=attn_mask)

        user_context = context[:, 0:1].expand_as(context[:, 1:])
        item_context = context[:, 1:]
        feed_mean_context = ((item_context * mask_f).sum(dim=1, keepdim=True) / counts).expand_as(item_context)

        final_repr = torch.cat((item_context, user_context, feed_mean_context, hist_context), dim=-1)
        return self.score(final_repr).squeeze(-1)


### fit  универсальный

In [7]:
import json
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm


def fit_v6(training, validation, config, output_dir, fine_tuning_models=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(config["threads"])
    torch.manual_seed(config["seed"])

    device, batch_size = config["device"], config["batch_size"]
    num_workers = config.get("num_workers", 0)
    use_amp = (device == "cuda")

    # Используем bfloat16 
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and amp_dtype == torch.float16))
    print(f"Режим точности AMP: {amp_dtype if use_amp else 'float32'}")

    #  ИНИЦИАЛИЗАЦИЯ МОДЕЛИ RankerV6 
    if fine_tuning_models is None:
        print("Обучение архитектуры RankerV6 с нуля!")
        model_params = {key: config[key] for key in ("width", "layers", "dropout")}
        model = RankerV6(len(training.categories_mapping), **model_params).to(device)
    else:
        print(f"Дообучение RankerV6 из {fine_tuning_models}...")
        checkpoint = torch.load(fine_tuning_models / "best_ndcg.pt", map_location=device, weights_only=False)
        model_params = checkpoint["model_params"]
        model = RankerV6(len(checkpoint["categories_mapping"]), **model_params).to(device)
        model.load_state_dict(checkpoint["state_dict"])

    # ОПТИМИЗАТОР 
    if config["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])

    val_loader = DataLoader(
        validation,
        batch_size=batch_size,
        collate_fn=collate_feeds_v6,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    #  ФУНКЦИЯ ПОТЕРЬ 
    loss_type = config["loss"]
    if loss_type == "Hybrid":
        weights = dict(config.get("loss_weights", {"bce": 0.1, "pairwise": 0.3, "lambda": 0.55}))
        criterion = HybridRankingLossV5(weights=weights, margin=0.5, k=30, sigma=1.0)
    elif loss_type in ("Lambda", "LambdaFeedLoss"):
        criterion = LambdaFeedLossV5(k=30, sigma=1.0, pos_alpha=0.25)
    elif loss_type == "Pairwise":
        criterion = PairwiseFeedLoss(margin=0.5)
    else:
        criterion = nn.BCEWithLogitsLoss()

    #  ШЕДУЛЕР И ДИНАМИЧЕСКИЕ ФРАКЦИИ 
    scheduler_type = config["scheduler"]
    if scheduler_type == "OneCycleLR" and fine_tuning_models is not None:
        print("OneCycleLR заменён на CosineAnnealingLR для безопасного дообучения.")
        scheduler_type = "CosineAnnealingLR"

    raw_fractions = config.get("epoch_data_fractions", 1.0)
    if isinstance(raw_fractions, (int, float)):
        fractions = [float(raw_fractions)] * config["epochs"]
    else:
        fractions = [float(x) for x in raw_fractions]

    g_subset = torch.Generator().manual_seed(config["seed"])

    total_steps = 0
    for ep in range(1, config["epochs"] + 1):
        frac = fractions[ep - 1] if ep <= len(fractions) else 1.0
        n_feeds = max(1, int(len(training) * frac)) if frac < 1.0 else len(training)
        total_steps += (n_feeds + batch_size - 1) // batch_size

    scheduler = None
    if scheduler_type == "OneCycleLR":
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=config["learning_rate"],
            total_steps=total_steps,
            pct_start=0.1,
            anneal_strategy="cos",
        )
    elif scheduler_type == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=config["scheduler_factor"], patience=config["scheduler_patience"]
        )
    elif scheduler_type == "CosineAnnealingLR":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config["epochs"], eta_min=1e-6
        )

    best_ndcg = -1.0
    best_loss = float("inf")
    history = []

    #  ЦИКЛ ОБУЧЕНИЯ ПО ЭПОХАМ 
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        train_loss = 0.0

        current_fraction = fractions[epoch - 1] if epoch <= len(fractions) else 1.0
        if current_fraction < 1.0:
            num_subset = max(1, int(len(training) * current_fraction))
            subset_indices = torch.randperm(len(training), generator=g_subset)[:num_subset].tolist()
            current_dataset = torch.utils.data.Subset(training, subset_indices)
            print(f"\n[Эпоха {epoch}] Обучение на части данных: {current_fraction*100:.0f}% ({num_subset} лент)")
        else:
            current_dataset = training
            print(f"\n[Эпоха {epoch}] Обучение на всём датасете: 100% ({len(training)} лент)")

        train_loader = DataLoader(
            current_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_feeds_v6,
            num_workers=num_workers,
            pin_memory=True,
        )
        pbar = tqdm(train_loader, desc=f"Эпоха {epoch}/{config['epochs']}")

        for batch in pbar:
            (
                features, category, users, cand_padding,
                hist_features, h_cats, h_etypes, h_esrcs, hist_padding,
                target, pos, _
            ) = (x.to(device, non_blocking=True) if x is not None else None for x in batch)

            optimizer.zero_grad(set_to_none=True)

            # Прямой проход в bfloat16 (без переполнения и NaN)
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                logits = model(
                    features, category, users, cand_padding,
                    hist_features, h_cats, h_etypes, h_esrcs, hist_padding
                )

            # Расчёт функции потерь строго в float32
            logits_f = logits.float()
            if loss_type == "Hybrid":
                loss_for_backward, raw_metrics = criterion(logits_f, target, cand_padding, pos=pos)
            elif loss_type in ("Lambda", "LambdaFeedLoss"):
                loss_for_backward = criterion(logits_f, target, cand_padding, pos=pos)
            elif loss_type == "Pairwise":
                loss_for_backward = criterion(logits_f, target, cand_padding)
            else:
                mask = ~cand_padding
                loss_for_backward = criterion(logits_f[mask], target[mask])

            # Защита от случайных выбросов в батче (не обновляем веса, если лосс NaN/Inf)
            if not torch.isfinite(loss_for_backward):
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss_for_backward).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            if loss_type == "Hybrid":
                display_loss = (
                    raw_metrics.get("bce", 0.0)
                    + raw_metrics.get("pair", 0.0)
                    + 0.01 * raw_metrics.get("lmb", 0.0)
                )
                train_loss += display_loss
                postfix = {k: f"{v:.4f}" if k != "lmb" else f"{v:.4f}" for k, v in raw_metrics.items()}
                pbar.set_postfix(postfix)
            else:
                train_loss += loss_for_backward.item()
                pbar.set_postfix({"loss": f"{loss_for_backward.item():.4f}"})

            if scheduler_type == "OneCycleLR" and scheduler is not None:
                scheduler.step()

        #  ВАЛИДАЦИЯ 
        model.eval()
        val_loss = 0.0
        val_preds, val_targets, val_feed_ids = [], [], []

        with torch.inference_mode():
            for batch in val_loader:
                (
                    features, category, users, cand_padding,
                    hist_features, h_cats, h_etypes, h_esrcs, hist_padding,
                    target, _, feed_ids
                ) = (x.to(device, non_blocking=True) if x is not None else None for x in batch)

                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                    logits = model(
                        features, category, users, cand_padding,
                        hist_features, h_cats, h_etypes, h_esrcs, hist_padding
                    )

                logits_f = logits.float()
                if loss_type == "Hybrid":
                    _, raw_metrics = criterion(logits_f, target, cand_padding, pos=None)
                    val_loss += (
                        raw_metrics.get("bce", 0.0)
                        + raw_metrics.get("pair", 0.0)
                        + 0.01 * raw_metrics.get("lmb", 0.0)
                    )
                elif loss_type in ("Lambda", "LambdaFeedLoss"):
                    val_loss += criterion(logits_f, target, cand_padding, pos=None).item()
                elif loss_type == "Pairwise":
                    val_loss += criterion(logits_f, target, cand_padding).item()
                else:
                    mask = ~cand_padding
                    val_loss += criterion(logits_f[mask], target[mask]).item()

                mask = ~cand_padding
                val_preds.append(logits_f[mask].detach().cpu().numpy())
                val_targets.append(target[mask].detach().cpu().numpy())
                lengths = mask.sum(dim=1).cpu().numpy()
                val_feed_ids.append(np.repeat(feed_ids.cpu().numpy(), lengths))

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        val_preds = np.concatenate(val_preds)
        val_targets = np.concatenate(val_targets)


        val_feed_ids = np.concatenate(val_feed_ids)

        ndcg = calculate_ndcg(val_targets, val_preds, val_feed_ids)

        metrics = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_ndcg": ndcg,
        }
        history.append(metrics)

        #  СОХРАНЕНИЕ МОДЕЛЕЙ 
        save_payload = {
            "state_dict": model.state_dict(),
            "categories_mapping": training.categories_mapping,
            "epoch": epoch,
            "model_params": model_params,
            "version": "v6",
            "val_ndcg": ndcg,
            "val_loss": val_loss,
        }

        if ndcg > best_ndcg:
            best_ndcg = ndcg
            torch.save(save_payload, output_dir / "best_ndcg.pt")
            print(f"  Сохранена модель по NDCG ({ndcg:.5f})")

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(save_payload, output_dir / "best_loss.pt")
            print(f" Сохранена модель по Val loss ({val_loss:.5f})")

        if scheduler is not None and scheduler_type != "OneCycleLR":
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(ndcg)
            else:
                scheduler.step()


        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Эпоха {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, "
            f"Val NDCG = {ndcg:.5f}, LR = {current_lr:.6f}"
        )

        torch.cuda.empty_cache()

    (output_dir / "history.json").write_text(json.dumps(history, indent=2))
    print("\n Обучение RankerV6 завершено!")
    return model


#  Тренировка ``` Ranker V6```

In [8]:
import json
from pathlib import Path
import numpy as np
import polars as pl
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

#  НАСТРОЙКИ 
DATA_DIR = Path("/data/develops/python/Tests/AVITO/competition_0")
MODELS_DIR = Path("/data/develops/python/Tests/AVITO/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

fine_tuning_models = MODELS_DIR / "HybridV6_2_01"
# fine_tuning_models = None

epoch_data = [0.1, 0.3, 0.5]

MAX_HIST_LEN = 48  

TRAIN_CONFIG_V6_LARGE = {
    "train_feeds": 0,
    "val_feeds": 0,
    "epochs": len(epoch_data),
    "batch_size": 256,            
    "learning_rate": 0.00009,     
    "weight_decay": 0.00015,
    "width": 192,                
    "layers": 4,                  
    "dropout": 0.08,
    "seed": 5789,                  
    "threads": 8,
    "num_workers": 0,
    "optimizer": "AdamW",
    "loss": "Hybrid",
    "loss_weights": {"bce": 0.05, "pairwise": 0.25, "lambda": 0.7},
    "scheduler": "OneCycleLR",          # "ReduceLROnPlateau", "CosineAnnealingLR", OneCycleLR, или None 
    "epoch_data_fractions": epoch_data, 
}
config = {**TRAIN_CONFIG_V6_LARGE, "device": DEVICE}


if "training" not in globals():
    print("Загружаем train V6...")
    training = CandidateDataV6(DATA_DIR / "train", config["train_feeds"])
if "validation" not in globals():
    print("Загружаем val V6...")
    validation = CandidateDataV6(DATA_DIR / "val", config["val_feeds"], training.categories_mapping)


print(f"Train лент: {len(training)}, Val лент: {len(validation)}")
model = fit_v6(training, validation, config, MODELS_DIR / "HybridV6_2_02", fine_tuning_models=fine_tuning_models)




Загружаем train V6...
Загружаем val V6...
Train лент: 3151208, Val лент: 14423
Режим точности AMP: torch.bfloat16
Дообучение RankerV6 из /data/develops/python/Tests/AVITO/models/HybridV6_2_01...
OneCycleLR заменён на CosineAnnealingLR для безопасного дообучения.

[Эпоха 1] Обучение на части данных: 10% (315120 лент)


Эпоха 1/3: 100%|██████████| 1231/1231 [10:55<00:00,  1.88it/s, bce=0.1005, pair=0.8257, lmb=0.1606]


  Сохранена модель по NDCG (0.31241)
 Сохранена модель по Val loss (0.86176)
Эпоха 1: Train Loss = 0.9182, Val Loss = 0.8618, Val NDCG = 0.31241, LR = 0.000068

[Эпоха 2] Обучение на части данных: 30% (945362 лент)


Эпоха 2/3: 100%|██████████| 3693/3693 [32:16<00:00,  1.91it/s, bce=0.1043, pair=0.8116, lmb=0.1613]


 Сохранена модель по Val loss (0.86052)
Эпоха 2: Train Loss = 0.9177, Val Loss = 0.8605, Val NDCG = 0.31175, LR = 0.000023

[Эпоха 3] Обучение на части данных: 50% (1575604 лент)


Эпоха 3/3: 100%|██████████| 6155/6155 [54:08<00:00,  1.89it/s, bce=0.0907, pair=0.8010, lmb=0.1621]


Эпоха 3: Train Loss = 0.9170, Val Loss = 0.8622, Val NDCG = 0.31197, LR = 0.000001

 Обучение RankerV6 завершено!


# Сабмит 

In [9]:
import json
from pathlib import Path
import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

#  НАСТРОЙКИ 
DATA_DIR = Path("/data/develops/python/Tests/AVITO/competition_0")
MODELS_DIR = Path("/data/develops/python/Tests/AVITO/models")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256

OUTPUT_DIR = MODELS_DIR / "HybridV6_2_02"
SUBMISSION_FILE = "submission_v6_2.parquet"

# ЗАГРУЗКА МОДЕЛИ RankerV6 
checkpoint = torch.load(OUTPUT_DIR / "best_ndcg.pt", map_location="cpu", weights_only=False)
mapping = checkpoint["categories_mapping"]

model = RankerV6(len(mapping), **checkpoint["model_params"]).to(DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Модель RankerV6 из {OUTPUT_DIR.name} успешно загружена! (Val NDCG = {checkpoint.get('val_ndcg', 0):.5f})")

#  ЗАГРУЗКА ТЕСТОВОГО ДАТАСЕТА 
print("Загружаем тестовый датасет V6...")
test_dataset = CandidateDataV6(DATA_DIR / "test", max_feeds=0, categories_mapping=mapping)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_feeds_v6, shuffle=False)
print(f"Тест загружен: {len(test_dataset)} лент.")

scores = []

# ИНФЕРЕНС 
with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Расчет скоров V6 на тесте"):
        # Из 12 элементов батча берём первые 9 входных тензоров для RankerV6
        (
            features,
            category,
            users,
            cand_padding,
            hist_features,
            h_cats,
            h_etypes,
            h_esrcs,
            hist_padding,
        ) = (x.to(DEVICE) for x in batch[:9])

        logits = model(
            features,
            category,
            users,
            cand_padding,
            hist_features,
            h_cats,
            h_etypes,
            h_esrcs,
            hist_padding,
        )

        # Берём сырые логиты без сигмоиды строго по маске ~cand_padding
        valid_logits = logits[~cand_padding].cpu().numpy()
        scores.extend(valid_logits)

scores = np.array(scores, dtype=np.float32)
assert len(scores) == len(test_dataset.keys), "Ошибка: количество предсказаний не совпадает с тестом!"
print(f"Рассчитано {len(scores)} предсказаний.")

# СОХРАНЕНИЕ САБМИТА 
submission = test_dataset.keys.with_columns(pl.Series("score", scores))
submission.write_parquet(SUBMISSION_FILE, compression="zstd")
print(f" Сабмит успешно сформирован и сохранен в {SUBMISSION_FILE}")


Модель RankerV6 из HybridV6_2_02 успешно загружена! (Val NDCG = 0.31241)
Загружаем тестовый датасет V6...
Тест загружен: 28595 лент.


Расчет скоров V6 на тесте: 100%|██████████| 112/112 [00:29<00:00,  3.78it/s]


Рассчитано 2904806 предсказаний.
 Сабмит успешно сформирован и сохранен в submission_v6_2.parquet


# END